# Step 5: Data Preprocessing, Anomaly Cleansing, and One-Hot Encoding

### 1 - DATASET INGESTION & OUTLIER ANOMALY RECONCILIATION
* **Our Solution:** Load checkpoint, impute 1 adult for ghost bookings, hard-cap ADR outliers at $500, and group rare countries.

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('bookings_with_holiday_features.csv')
ghost_mask = (df['adults'] + df['children'] + df['babies']) == 0
df.loc[ghost_mask, 'adults'] = 1
df = df[df['adr'] >= 0].copy()
df.loc[df['adr'] > 500.0, 'adr'] = 500.0
country_counts = df['country'].value_counts(dropna=False)
rare_countries = country_counts[country_counts < 10].index
df['country'] = df['country'].replace(rare_countries, 'Other')

### 2 - EXPLICIT NULL ALLOCATION & ONE-HOT ENCODING

In [ ]:
y = df['is_canceled'].copy()
features_to_drop = ['is_canceled', 'booking_date', 'arrival_date', 'departure_date']
X_raw = df.drop(columns=features_to_drop)
X_raw['country'] = X_raw['country'].fillna('Unknown')
X_raw['agent'] = X_raw['agent'].fillna(-1).astype('int64')
categorical_columns = ['hotel', 'deposit_type', 'market_segment', 'meal', 'customer_type', 'distribution_channel', 'reserved_room_type', 'country']
X = pd.get_dummies(X_raw, columns=categorical_columns, drop_first=True)
bool_columns = X.select_dtypes(include=['bool']).columns
X[bool_columns] = X[bool_columns].astype(int)

### 3 - CHRONOLOGICAL TEMPORAL SPLITTING

In [ ]:
arrival_dates_series = pd.to_datetime(df['arrival_date'])
cutoff_boundary = arrival_dates_series.min() + pd.DateOffset(months=18)
train_mask = arrival_dates_series < cutoff_boundary
test_mask = arrival_dates_series >= cutoff_boundary
X_train, X_test = X[train_mask].copy(), X[test_mask].copy()
y_train, y_test = y[train_mask].copy(), y[test_mask].copy()

### 4 - ALGORITHM BENCHMARKING (Champion vs. Challenger)
Results show HistGradientBoosting (ROC-AUC 0.9032) outperforms Random Forest (0.8818).

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score, log_loss
import time
models = {
    "Random Forest (Baseline)": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "HistGradientBoosting (Advanced)": HistGradientBoostingClassifier(max_iter=150, max_depth=6, learning_rate=0.08, random_state=42)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    print(f"{name}: ROC-AUC={roc_auc_score(y_test, probs):.4f}, Log-Loss={log_loss(y_test, probs):.4f}")
xgb_model = models["HistGradientBoosting (Advanced)"]

### 5 - FEATURE IMPORTANCE & DATA LEAKAGE REMEDIATION
After discovering leakage in `segment_cancels_on_booking_date` and `segment_bookings_made_on_booking_date`, these were removed to ensure operational validity.

In [ ]:
leakage_cols = ['segment_cancels_on_booking_date', 'segment_bookings_made_on_booking_date']
X_clean = X.drop(columns=[col for col in leakage_cols if col in X.columns])
X_train_clean, X_test_clean = X_clean[train_mask], X_clean[test_mask]
xgb_model.fit(X_train_clean, y_train)
print("Model re-calibrated successfully without leakage variables.")

### 6 - OPERATIONAL PORTFOLIO ETL PIPELINE (TIME MACHINE)

In [ ]:
booking_dates, arrival_dates = pd.to_datetime(df['booking_date']), pd.to_datetime(df['arrival_date'])
simulation_days = pd.date_range(start='2015-10-01', end='2017-08-15', freq='D')
bi_export_list = []
for current_date in simulation_days:
    mask = (booking_dates <= current_date) & (arrival_dates > current_date) & (arrival_dates <= current_date + pd.Timedelta(days=30))
    sim_idx = df[mask].index
    if len(sim_idx) == 0: continue
    probs = xgb_model.predict_proba(X_clean.loc[sim_idx])[:, 1]
    df_slice = df.loc[sim_idx].copy()
    df_slice['raw_prob'], df_slice['standing_date'] = probs, current_date
    df_slice['exposure_score'] = df_slice['raw_prob'].clip(0, 1)
    bi_export_list.append(df_slice[['standing_date', 'arrival_date', 'is_canceled', 'raw_prob', 'exposure_score', 'hotel', 'market_segment']])
final_bi_dataset = pd.concat(bi_export_list, ignore_index=True)
final_bi_dataset.to_csv('bi_dashboard_pipeline_output.csv', index=False)
print(f"Exported {final_bi_dataset.shape[0]:,} records.")